# PPO DrugGPT GPCR Experiment

This notebook ports the GPCR-conditioned DrugGPT experiment from GRPO to TRL's current PPO trainer.

It keeps the same prompt format, the same RDKit plus XGBoost reward oracle, and a similar artifact layout while adapting training and metrics collection to PPO.

## 1) Environment and PPO dependency bootstrap

Run the next cell once in the `.venv` kernel. It verifies the core packages and ensures a TRL build with the documented PPO trainer API is available.

If you are on Apple Silicon and PPO rollouts are unstable on MPS, this notebook falls back to CPU for PPO training.

In [1]:
import importlib
import platform
import shutil
import subprocess
import sys
from pathlib import Path

REQUIRED = {
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
    "accelerate": "accelerate",
    "peft": "peft",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "tqdm": "tqdm",
    "xgboost": "xgboost",
    "dotenv": "python-dotenv",
    "packaging": "packaging",
    "trl": "trl[peft]>=1.1.0",
}
OPTIONAL = {
    "rdkit": "rdkit",
}

UV_BIN = shutil.which("uv")


def in_virtual_env() -> bool:
    return sys.prefix != getattr(sys, "base_prefix", sys.prefix)


def build_install_cmd(package_name: str):
    if UV_BIN is not None:
        return [UV_BIN, "pip", "install", "--python", sys.executable, package_name], "uv pip"
    return [sys.executable, "-m", "pip", "install", package_name], "pip"


def ensure_pkg(import_name: str, pip_name: str | None = None, optional: bool = False):
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
        return True, "already installed"
    except Exception:
        if optional:
            return False, "optional and missing"
        cmd, installer = build_install_cmd(pip_name)
        try:
            subprocess.check_call(cmd)
            return True, f"installed via {installer}"
        except Exception as exc:
            return False, f"failed install via {installer}: {exc}"


print("Python:", sys.version)
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Virtual env active:", in_virtual_env())
print("uv detected:", UV_BIN is not None)
if UV_BIN:
    print("uv binary:", UV_BIN)

results = {}
for import_name, pip_name in REQUIRED.items():
    ok, msg = ensure_pkg(import_name, pip_name=pip_name)
    results[import_name] = (ok, msg)

for import_name, pip_name in OPTIONAL.items():
    ok, msg = ensure_pkg(import_name, pip_name=pip_name, optional=True)
    results[import_name] = (ok, msg)

print("\nDependency status:")
for pkg, (ok, msg) in results.items():
    print(f"- {pkg}: {'OK' if ok else 'MISSING'} ({msg})")

Python: 3.12.11 (main, Aug  8 2025, 16:50:44) [Clang 20.1.4 ]
Executable: /Users/sahilkabir/Documents/School/DrugGenGPCR/.venv/bin/python
Platform: macOS-15.6.1-arm64-arm-64bit
Virtual env active: True
uv detected: True
uv binary: /Users/sahilkabir/.local/bin/uv


/Users/sahilkabir/Documents/School/DrugGenGPCR/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Dependency status:
- torch: OK (already installed)
- transformers: OK (already installed)
- datasets: OK (already installed)
- accelerate: OK (already installed)
- peft: OK (already installed)
- sklearn: OK (already installed)
- pandas: OK (already installed)
- numpy: OK (already installed)
- matplotlib: OK (already installed)
- seaborn: OK (already installed)
- tqdm: OK (already installed)
- xgboost: OK (already installed)
- dotenv: OK (already installed)
- packaging: OK (already installed)
- trl: OK (already installed)
- rdkit: OK (already installed)


In [2]:
import json
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import transformers
import xgboost as xgb
from datasets import Dataset
from dotenv import load_dotenv
from packaging.version import Version
from peft import LoraConfig
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer, TrainerCallback
from trl.experimental.ppo import PPOConfig, PPOTrainer

load_dotenv()
RDLogger.DisableLog("rdApp.*")


if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# PPO rollout generation is still more predictable on CPU than on MPS.
# ppo_device = torch.device("cpu") if device.type == "mps" else device
ppo_device = device
policy_dtype = torch.bfloat16 if ppo_device.type == "cuda" else torch.float32

print("Selected device:", device)
print("PPO trainer device:", ppo_device)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("xgboost:", xgb.__version__)
if device.type == "mps":
    print("Using CPU for PPO to avoid Apple Silicon rollout instability.")

Selected device: mps
PPO trainer device: mps
torch: 2.11.0
transformers: 5.5.4
xgboost: 3.2.0
Using CPU for PPO to avoid Apple Silicon rollout instability.


/var/folders/cw/_ym5rb7s2dnflr0tgzm6b4k00000gn/T/ipykernel_24539/1081841754.py:22: TRLExperimentalWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  from trl.experimental.ppo import PPOConfig, PPOTrainer


## 2) Data loading and 80/20 split

This matches the GRPO notebook: same CSV, same prompt shape, same class filtering, and the same stratified split.

In [3]:
DATA_PATH = Path("gpcr_class_sequences_from_json.csv")
assert DATA_PATH.exists(), "Expected gpcr_class_sequences_from_json.csv in repo root"


def normalize_prompt_key(text: str) -> str:
    return text.strip()


df = pd.read_csv(DATA_PATH)
required_cols = {"sequence", "gpcr_binding_encoded"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df = df.dropna(subset=["sequence", "gpcr_binding_encoded"]).copy()
df["gpcr_binding_encoded"] = df["gpcr_binding_encoded"].astype(int)
df["prompt"] = df["sequence"].map(lambda s: f"<|startoftext|><P>{s}<L>")
df = df.groupby("gpcr_binding_encoded").filter(lambda x: len(x) >= 3).reset_index(drop=True)

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["gpcr_binding_encoded"],
)

prompt_label_counts = df.groupby("prompt")["gpcr_binding_encoded"].nunique()
conflicting_prompts = prompt_label_counts[prompt_label_counts > 1]
if not conflicting_prompts.empty:
    raise ValueError("A prompt maps to multiple GPCR classes in the filtered dataset; the PPO reward lookup would be ambiguous.")

prompt_to_label = (
    df[["prompt", "gpcr_binding_encoded"]]
    .drop_duplicates(subset=["prompt"])
    .assign(prompt=lambda x: x["prompt"].map(normalize_prompt_key))
    .set_index("prompt")["gpcr_binding_encoded"]
    .astype(int)
    .to_dict()
)

print("Total rows:", len(df))
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Classes in train:", train_df["gpcr_binding_encoded"].nunique())
print("Classes in test:", test_df["gpcr_binding_encoded"].nunique())
print("Prompt-to-label mappings:", len(prompt_to_label))
print(train_df["gpcr_binding_encoded"].value_counts().sort_index())

Total rows: 1109
Train rows: 887
Test rows: 222
Classes in train: 78
Classes in test: 72
Prompt-to-label mappings: 1083
gpcr_binding_encoded
0      3
1     67
2     23
5      4
6      3
      ..
83     6
84     4
85     6
86    12
87     6
Name: count, Length: 78, dtype: int64


## 3) Reward oracle and PPO adapter

The scoring logic below keeps the original SMILES extraction and XGBoost margin reward intact.

The only structural change is the adapter layer at the end: PPO expects a reward model module, so the notebook wraps the existing sequence-level oracle in a lightweight module that decodes each prompt plus completion pair and emits a scalar score.

In [4]:
MODEL_PATH = Path("xgb_rdkit_classify_fp.json")
if not MODEL_PATH.exists():
    print("WARNING: xgb_rdkit_classify_fp.json not found in repo root.")
    print("The reward model will assign the invalid-SMILES penalty until MODEL_PATH is fixed.")

xgb_model = xgb.XGBClassifier(
    nthread=1,
    predictor="cpu_predictor",
)
if MODEL_PATH.exists():
    xgb_model.load_model(str(MODEL_PATH))
    dummy = np.zeros((3, 2048), dtype=np.float32)
    _ = xgb_model.predict_proba(dummy)
    print("XGBoost model loaded and dummy inference passed.")

INVALID_SMILES_PENALTY = -0.5


@dataclass
class RewardResult:
    reward: float
    is_valid_smiles: bool
    pred_class: int
    confidence: float
    smiles: str


def smiles_to_fp(smiles: str, n_bits: int = 2048, radius: int = 3):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    bit_vect = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(bit_vect, arr)
    return arr


def _first_token(s: str) -> str:
    s = s.strip()
    if not s:
        return ""
    parts = s.split()
    return parts[0] if parts else ""


def extract_smiles(decoded_text: str) -> str:
    if not decoded_text:
        return ""
    if "<L>" in decoded_text:
        tail = decoded_text.split("<L>", 1)[1].strip()
        return _first_token(tail)
    return _first_token(decoded_text)


def extract_prompt_key(decoded_text: str) -> str:
    if not decoded_text or "<L>" not in decoded_text:
        return ""
    return normalize_prompt_key(decoded_text.split("<L>", 1)[0] + "<L>")


def reward_from_decoded(decoded_text: str, true_class: int):
    smiles = extract_smiles(decoded_text)
    fp = smiles_to_fp(smiles)
    if fp is None or (not MODEL_PATH.exists()):
        return RewardResult(
            reward=INVALID_SMILES_PENALTY,
            is_valid_smiles=False,
            pred_class=-1,
            confidence=0.0,
            smiles=smiles,
        )

    X = fp.reshape(1, -1)
    logits = xgb_model.predict(X, output_margin=True)[0]
    pred_class = int(np.argmax(logits))
    conf_true = float(logits[int(true_class)]) if int(true_class) < len(logits) else 0.0
    reward = float(logits[int(true_class)] - max([logits[i] for i in range(len(logits)) if i != int(true_class)]))

    return RewardResult(
        reward=reward,
        is_valid_smiles=True,
        pred_class=pred_class,
        confidence=conf_true,
        smiles=smiles,
    )


def reward_from_text(decoded_text: str) -> RewardResult:
    prompt_key = extract_prompt_key(decoded_text)
    true_class = prompt_to_label.get(prompt_key)
    if true_class is None:
        return RewardResult(
            reward=INVALID_SMILES_PENALTY,
            is_valid_smiles=False,
            pred_class=-1,
            confidence=0.0,
            smiles=extract_smiles(decoded_text),
        )
    return reward_from_decoded(decoded_text, int(true_class))


class OracleRewardBackbone(nn.Module):
    def __init__(self, tokenizer, owner):
        super().__init__()
        self.tokenizer = tokenizer
        self.owner = owner

    def forward(self, input_ids, attention_mask=None, **kwargs):
        if attention_mask is None:
            attention_mask = input_ids.ne(self.owner.pad_token_id)

        batch_size, seq_len = input_ids.shape
        hidden = torch.zeros((batch_size, seq_len, 1), device=input_ids.device, dtype=torch.float32)
        batch_results = []

        for row_idx in range(batch_size):
            row_mask = attention_mask[row_idx].bool()
            token_ids = input_ids[row_idx][row_mask].detach().cpu().tolist()
            decoded_text = self.tokenizer.decode(token_ids, skip_special_tokens=False)
            rr = reward_from_text(decoded_text)
            hidden[row_idx, row_mask, 0] = float(rr.reward)
            batch_results.append(rr)

        if batch_results:
            invalid_rate = 1.0 - (sum(1.0 if rr.is_valid_smiles else 0.0 for rr in batch_results) / len(batch_results))
            confidence_mean = float(np.mean([rr.confidence for rr in batch_results]))
            reward_mean = float(np.mean([rr.reward for rr in batch_results]))
        else:
            invalid_rate = 0.0
            confidence_mean = 0.0
            reward_mean = 0.0

        self.owner.last_batch_stats = {
            "oracle/invalid_rate": invalid_rate,
            "oracle/confidence_mean": confidence_mean,
            "oracle/reward_mean": reward_mean,
        }
        self.owner.history.append(self.owner.last_batch_stats.copy())
        return SimpleNamespace(hidden_states=(hidden,))


class OracleRewardModel(nn.Module):
    base_model_prefix = "backbone"

    def __init__(self, tokenizer, pad_token_id: int):
        super().__init__()
        self.pad_token_id = pad_token_id
        self.last_batch_stats = {}
        self.history = []
        self.backbone = OracleRewardBackbone(tokenizer=tokenizer, owner=self)
        self.score = nn.Identity()
        self.register_buffer("_anchor", torch.zeros(1), persistent=False)


class OracleStatsCallback(TrainerCallback):
    def __init__(self, oracle_model: OracleRewardModel):
        self.oracle_model = oracle_model

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or not self.oracle_model.last_batch_stats:
            return control
        logs.update(self.oracle_model.last_batch_stats)
        if state.log_history:
            state.log_history[-1].update(self.oracle_model.last_batch_stats)
        return control

: 

## 4) Policy, value model, datasets, and PPO trainer

This follows the current TRL PPO structure more closely than the GRPO notebook:

- policy: `AutoModelForCausalLM`
- value model: `AutoModelForSequenceClassification(num_labels=1)`
- reward model: the custom oracle wrapper above
- PEFT: LoRA on the policy model only

In [ ]:
MODEL_NAME = "liyuesen/druggpt"
OUTPUT_DIR = Path("outputs/ppo_druggpt")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAINER_OUTPUT_DIR = OUTPUT_DIR / "trl_ppo"
TRAINER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

response_length = 64
per_device_train_batch_size = 1
gradient_accumulation_steps = 4
local_rollout_forward_batch_size = 1 if ppo_device.type != "cuda" else 4


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("Using EOS token as PAD token for PPO batching.")

policy = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=policy_dtype)
value_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    torch_dtype=policy_dtype,
)

for model_obj in [policy, value_model]:
    model_obj.config.pad_token_id = tokenizer.pad_token_id
    if hasattr(model_obj, "generation_config"):
        model_obj.generation_config.pad_token_id = tokenizer.pad_token_id
        model_obj.generation_config.eos_token_id = tokenizer.eos_token_id

reward_model = OracleRewardModel(tokenizer=tokenizer, pad_token_id=tokenizer.pad_token_id)
peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["c_attn", "c_proj", "c_fc"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)
ref_policy = None


def prepare_prompt_dataset(frame: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(frame[["prompt"]].reset_index(drop=True), preserve_index=False)

    def tokenize(batch):
        outputs = tokenizer(batch["prompt"], padding=False)
        return {
            "input_ids": outputs["input_ids"],
            "lengths": [len(ids) for ids in outputs["input_ids"]],
        }

    ds = ds.map(tokenize, batched=True, remove_columns=ds.column_names)
    ds = ds.filter(lambda row: row["lengths"] <= 512)
    return ds.remove_columns(["lengths"])


train_dataset = prepare_prompt_dataset(train_df)
eval_limit = min(len(test_df), 64)
eval_dataset = prepare_prompt_dataset(test_df.head(eval_limit)) if eval_limit else None

assert train_dataset[0]["input_ids"][-1] != tokenizer.eos_token_id, "Prompt should end at <L>, not EOS."

ppo_config = PPOConfig(
    output_dir=str(TRAINER_OUTPUT_DIR),
    sft_model_path=MODEL_NAME,
    reward_model_path=str(MODEL_PATH),
    learning_rate=1e-6,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    total_episodes=max(len(train_dataset), 256),
    num_ppo_epochs=1,
    num_mini_batches=1,
    local_rollout_forward_batch_size=local_rollout_forward_batch_size,
    response_length=response_length,
    stop_token="eos",
    missing_eos_penalty=0.25,
    temperature=0.7,
    kl_coef=0.03,
    whiten_rewards=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    report_to=[],
    bf16=(ppo_device.type == "cuda"),
    use_cpu=(ppo_device.type == "cpu"),
)

trainer = PPOTrainer(
    args=ppo_config,
    processing_class=tokenizer,
    model=policy,
    ref_model=ref_policy,
    reward_model=reward_model,
    value_model=value_model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    callbacks=[OracleStatsCallback(reward_model)],
)

print("Train prompts:", len(train_dataset))
print("Eval prompts:", 0 if eval_dataset is None else len(eval_dataset))
print("Trainer output dir:", TRAINER_OUTPUT_DIR.resolve())
print("Ready to train with PPOTrainer.")

## 5) Train PPO and collect artifacts

The PPO logs will expose RLHF-specific metrics such as score, KL, and policy/value loss.

The final cell also runs a lightweight evaluation pass on held-out GPCR prompts to track reward quality and invalid-SMILES rate with the same oracle used for training.

In [ ]:
print("Starting PPOTrainer training...")
train_result = trainer.train()
print(train_result)

trainer.save_model(str(TRAINER_OUTPUT_DIR / "final"))
tokenizer.save_pretrained(TRAINER_OUTPUT_DIR / "final")


def unwrap_policy_model(trainer_obj):
    model_obj = trainer_obj.accelerator.unwrap_model(trainer_obj.model)
    return model_obj.policy if hasattr(model_obj, "policy") else model_obj


def generate_decoded(prompt: str, model, max_new_tokens: int = 64, deterministic: bool = False):
    generation_device = next(model.parameters()).device
    encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    input_ids = encoded["input_ids"].to(generation_device)
    attention_mask = encoded["attention_mask"].to(generation_device)

    with torch.no_grad():
        output = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            do_sample=not deterministic,
            temperature=0.7 if not deterministic else 1.0,
            top_p=1.0,
            top_k=0,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    completion_ids = output[0, input_ids.shape[1]:]
    completion = tokenizer.decode(completion_ids, skip_special_tokens=True)
    return prompt + completion


policy_for_generation = unwrap_policy_model(trainer)
policy_for_generation.eval()

sample_eval_df = test_df.head(min(len(test_df), 32)).copy()
samples = []
for row in sample_eval_df.itertuples(index=False):
    decoded = generate_decoded(row.prompt, policy_for_generation, max_new_tokens=response_length, deterministic=False)
    rr = reward_from_decoded(decoded, int(row.gpcr_binding_encoded))
    samples.append(
        {
            "prompt": row.prompt,
            "true_class": int(row.gpcr_binding_encoded),
            "decoded_text": decoded,
            "generated_smiles": rr.smiles,
            "pred_class": rr.pred_class,
            "confidence": rr.confidence,
            "reward": rr.reward,
            "is_valid_smiles": rr.is_valid_smiles,
        }
    )

generations_df = pd.DataFrame(samples)
generations_df.to_csv(OUTPUT_DIR / "sample_generations.csv", index=False)

metrics_df = pd.DataFrame(trainer.state.log_history)
metrics_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)

if "uniprot_id" in train_df.columns:
    train_df[["uniprot_id"]].reset_index(drop=True).to_csv(OUTPUT_DIR / "train_ids.csv", index=False)
if "uniprot_id" in test_df.columns:
    test_df[["uniprot_id"]].reset_index(drop=True).to_csv(OUTPUT_DIR / "test_ids.csv", index=False)

oracle_history_df = pd.DataFrame(reward_model.history)
oracle_history_df.to_csv(OUTPUT_DIR / "oracle_reward_batches.csv", index=False)

eval_summary = {
    "sample_count": int(len(generations_df)),
    "reward_mean": float(generations_df["reward"].mean()) if not generations_df.empty else None,
    "invalid_rate": float(1.0 - generations_df["is_valid_smiles"].mean()) if not generations_df.empty else None,
    "confidence_mean": float(generations_df["confidence"].mean()) if not generations_df.empty else None,
}

hparams = {
    "model_name": MODEL_NAME,
    "trl_version": trl.__version__,
    "reward_backend": "rdkit_plus_xgboost_margin",
    "invalid_smiles_penalty": INVALID_SMILES_PENALTY,
    "response_length": response_length,
    "per_device_train_batch_size": ppo_config.per_device_train_batch_size,
    "gradient_accumulation_steps": ppo_config.gradient_accumulation_steps,
    "num_ppo_epochs": ppo_config.num_ppo_epochs,
    "total_episodes": ppo_config.total_episodes,
    "learning_rate": ppo_config.learning_rate,
    "kl_coef": ppo_config.kl_coef,
    "missing_eos_penalty": ppo_config.missing_eos_penalty,
    "device": str(device),
    "ppo_device": str(ppo_device),
    "eval_summary": eval_summary,
}
(OUTPUT_DIR / "hparams.json").write_text(json.dumps(hparams, indent=2))

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
axes = axes.flatten()

x_col = "episode" if "episode" in metrics_df.columns else ("step" if "step" in metrics_df.columns else None)
if x_col is None or metrics_df.empty:
    print("No PPO metrics available yet for plotting.")
else:
    plot_specs = [
        ("objective/rlhf_reward", "RLHF Reward"),
        ("objective/scores", "Oracle Score"),
        ("objective/kl", "KL"),
        ("oracle/invalid_rate", "Invalid SMILES Rate"),
        ("loss/policy_avg", "Policy Loss"),
        ("loss/value_avg", "Value Loss"),
    ]
    for ax, (column, title) in zip(axes, plot_specs, strict=True):
        if column in metrics_df.columns:
            sub = metrics_df[[x_col, column]].dropna().copy()
            if not sub.empty:
                sub[f"{column}_smooth"] = sub[column].rolling(window=5, min_periods=1).mean()
                ax.plot(sub[x_col], sub[column], alpha=0.3, label="raw")
                ax.plot(sub[x_col], sub[f"{column}_smooth"], label="smooth")
        ax.set_title(title)
        ax.set_xlabel(x_col)
        ax.legend(loc="best")

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "training_curves.png", dpi=200)
    plt.show()

print("Reproducibility summary")
print("- model:", MODEL_NAME)
print("- train rows:", len(train_df))
print("- test rows:", len(test_df))
print("- output dir:", OUTPUT_DIR.resolve())
print("- sample eval summary:", eval_summary)